# Evaluation: LLMs and BERT Models on 16 Hate Speech Test Sets

This notebook evaluates fine-tuned LLM models (Llama3.2-1B and Qwen2.5-14B) and BERT-based models
on 16 benchmark hate speech test sets across English, German, Spanish, and Vietnamese.

**Evaluation tracks:**
- **LLM track**: Load each fine-tuned model variant (Human, LGB, Mean, Vote, Human-LGB), run inference
  using token probability extraction, and compute macro-F1 per dataset.
- **BERT track**: Fine-tune or load a pre-fine-tuned BERT model, run inference, and generate a per-dataset report.

Pre-computed probability outputs are saved as `.pkl` files in this folder for reproducibility.

## 1. Imports and Configuration

Import all required libraries, set the random seed for reproducibility, and configure the device
(CUDA if available). The `FastLanguageModel` class from Unsloth is used for memory-efficient
4-bit quantized inference.

In [6]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments, 
    Trainer,
)
from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch
from datasets import load_dataset
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)
import torch.nn.functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'


/home/dangdang/miniconda3/envs/ros_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1 LLMs

Selecting batch size

In [ ]:
batch_size = 64

Zero-shot prompt tempalte for 2 label task: Hate + Neutral

In [7]:
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Normal Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
\n"{comment}"\n
Your Answer:
'''



A list of fine-tuned models is available on Hugging Face

In [8]:
models_list_HF = {"Llama3.2-1B": {
                    "Base": "unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit",
                    "Human": "danghaidang-passau/Hate-Llama3.2-1B.human.2_label",
                    "Lgb": "danghaidang-passau/Hate-Llama3.2-1B.Lgb.2_label",
                    "Mean": "danghaidang-passau/Hate-Llama3.2-1B.Mean.2_label",
                    "Vote": "danghaidang-passau/Hate-Llama3.2-1B.Vote.2_label",
                    "Human-Lgb": "danghaidang-passau/Hate-Llama3.2-1B.Human_Lgb.2_label",
                    },
                "Qwen2.5-14B": {
                    "Base": "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                    "Human": "danghaidang-passau/Hate-Qwen2.5-14B.Human.2_label",
                    "Lgb": "danghaidang-passau/Hate-Qwen2.5-14B.Lgb.2_label",
                    "Mean": "danghaidang-passau/Hate-Qwen2.5-14B.Mean.2_label",
                    "Vote": "danghaidang-passau/Hate-Qwen2.5-14B.Vote.2_label",
                    "Human-Lgb": "danghaidang-passau/Hate-Qwen2.5-14B.Human_Lgb.2_label",
                    }}


In [7]:
repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_16 = load_dataset(repo, "16-val-train")       
ds_train = ds_16["train"]
ds_val = ds_16["test"]

Evaluation set

In [8]:
df_train = ds_train.to_pandas()
df_val = ds_val.to_pandas()

In [12]:
df_val.ds.value_counts()

ds
HateXplain      3846
GermEval18      3532
AHSD            3000
ViHSD           2672
Sexism          2632
GermEval19      2507
Gahd            2198
GermEval21      2085
Chileno         1928
HateEval-spa    1286
Haternet        1205
US_election     1117
HateEval-eng    1000
Covid            971
AbusEval         860
HASOC            526
Name: count, dtype: int64

In [13]:
print(df_val.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

label_id         1     2
ds                      
AHSD          2494   506
AbusEval       178   682
Chileno        110  1818
Covid          190   781
Gahd           939  1259
GermEval18    1202  2330
GermEval19     840  1667
GermEval21     769  1316
HASOC          134   392
HateEval-eng   427   573
HateEval-spa   556   730
HateXplain    2283  1563
Haternet       305   900
Sexism         350  2282
US_election    140   977
ViHSD          482  2190


Selecting Group model Llama3.2-1B or Qwen2.5-14B 
Test set: 1 or 2

In [14]:
model_group = "Llama3.2-1B"

## 3. Inference Helper Functions

**`process_task`**: Takes a batch of prompted texts, runs them through a quantized LLM, and extracts
the softmax probabilities for the two stop tokens (the "1" and "2" label tokens). Returns per-sample
probability arrays for each label.

**`preprocess`**: Applies the chat template to a raw text string using the model's tokenizer.
Handles model-specific system prompts (Qwen uses an Alibaba Cloud assistant identity; Llama and
others use a generic assistant prompt).

**`run_eval`**: Main evaluation loop. Iterates over all model variants for the selected `model_group`,
loads each model in 4-bit precision, applies the chat template to all test texts, batches inference,
and stores per-sample label probabilities in a nested dictionary keyed by model variant.

In [16]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]
    last_token_logits = logits[:, -1, :]  # Shape: [vocab_size]
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    selected_probs_1 = probabilities[:, indices[0]].float().cpu().numpy()
    selected_probs_2 = probabilities[:, indices[1]].float().cpu().numpy()
    return selected_probs_1, selected_probs_2

In [15]:
def preprocess(text, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    messages = [system_message, user_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages


    return messages


In [3]:
def run_eval():
    model_probs_dict = {}

    model_list = models_list_HF[model_group]
    df_eval = df_val


    for key, model_id in model_list.items():

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=500,
            dtype=getattr(torch, "bfloat16"),
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side = "left"

        stop_token_id = [16, 17]

        if model_group == "Qwen2.5-14B":
            stop_token_id = tokenizer(["12"])['input_ids'][0]
        elif model_group == "Llama3.2-1B":
            stop_token_id = [16, 17]


        df_eval["prompt"] = df_eval["text"].apply(lambda text: preprocess(text, model_id, tokenizer))


        texts = []
        probs_label_1 = []
        probs_label_2 = []

        prompts = df_eval['prompt'].tolist()

        for i in tqdm(range(0, len(prompts), batch_size)):
            batch = prompts[i:i+batch_size]
            selected_probs_1, selected_probs_2 = process_task(batch, model, tokenizer, stop_token_id)
            probs_label_1.extend(selected_probs_1.tolist())
            probs_label_2.extend(selected_probs_2.tolist())
            torch.cuda.empty_cache()
            torch.cuda.synchronize()


        model_probs_dict[key] = {
            "probs_label_1": probs_label_1,
            "probs_label_2": probs_label_2,
        }
    return model_probs_dict

def make_report(model_probs_dict, df_eval):
    report = {}
    for ds in df_eval['ds'].unique():
        report[ds] = {}
    report["Mean"] = {}

    y_true = np.array(df_eval['label_id'] == 1, dtype=int)
    df_eval['y_ture'] = y_true

    for model, probs in model_probs_dict.items():
        y_prob_label_1 = probs["probs_label_1"]
        df_eval['y_pred'] = y_prob_label_1 > np.mean(y_prob_label_1)
        df_eval['probs'] = y_prob_label_1

        report["Mean"][model] = {
        "acc": round(accuracy_score(y_true, df_eval['y_pred'])* 100, 1), 
        "f1": round(f1_score(y_true,df_eval['y_pred'], average='macro')* 100, 1)
            }
        
        for ds in df_eval['ds'].unique():
            tmp_df = df_eval.loc[df_eval['ds'] == ds]

            y_pred = tmp_df['y_pred'] 


            acc = round(accuracy_score(tmp_df['y_ture'],y_pred)* 100, 1) 
            f1 = round(f1_score(tmp_df['y_ture'], y_pred, average='macro')* 100, 1) 
            report[ds][model] = {
                            "acc": acc, 
                            "f1": f1
                            }
    
    report_df = pd.DataFrame(report).T
    return report_df

## 4. Run Evaluation: Llama3.2-1B

Run evaluation for all Llama3.2-1B model variants. Results are stored in `probs`.
If you want to skip re-running inference, load the pre-computed probabilities from
`Llama1B_probs.pkl` (already included in this folder) and call `make_report` directly.

In [ ]:
model_group = "Llama3.2-1B"
result = {}
probs = run_eval()

In [2]:
import pickle
with open("Llama1B_probs.pkl", "rb") as f:
    probs = pickle.load(f)

In [9]:
report = make_report(probs, df_val)

In [10]:
report

,Base,Human,Lgb,Mean,Vote,Human-Lgb
HateXplain,"{'acc': 53.6, 'f1': 53.3}","{'acc': 58.4, 'f1': 58.4}","{'acc': 67.7, 'f1': 67.0}","{'acc': 60.8, 'f1': 60.7}","{'acc': 61.3, 'f1': 61.3}","{'acc': 70.4, 'f1': 67.3}"
GermEval18,"{'acc': 42.5, 'f1': 40.0}","{'acc': 44.7, 'f1': 42.8}","{'acc': 68.6, 'f1': 67.2}","{'acc': 55.0, 'f1': 54.9}","{'acc': 53.5, 'f1': 53.3}","{'acc': 58.8, 'f1': 58.7}"
Haternet,"{'acc': 35.3, 'f1': 34.9}","{'acc': 44.2, 'f1': 44.0}","{'acc': 52.3, 'f1': 49.8}","{'acc': 38.6, 'f1': 38.5}","{'acc': 36.3, 'f1': 35.9}","{'acc': 50.9, 'f1': 50.5}"
AHSD,"{'acc': 44.0, 'f1': 40.8}","{'acc': 41.5, 'f1': 40.5}","{'acc': 51.7, 'f1': 49.7}","{'acc': 44.4, 'f1': 43.6}","{'acc': 45.5, 'f1': 44.5}","{'acc': 73.6, 'f1': 66.3}"
AbusEval,"{'acc': 64.5, 'f1': 56.2}","{'acc': 75.9, 'f1': 52.7}","{'acc': 80.3, 'f1': 66.1}","{'acc': 77.7, 'f1': 51.5}","{'acc': 77.7, 'f1': 52.2}","{'acc': 78.7, 'f1': 61.7}"
GermEval19,"{'acc': 42.8, 'f1': 40.8}","{'acc': 44.8, 'f1': 43.5}","{'acc': 67.9, 'f1': 66.8}","{'acc': 55.4, 'f1': 55.4}","{'acc': 53.3, 'f1': 53.1}","{'acc': 59.2, 'f1': 59.1}"
ViHSD,"{'acc': 50.8, 'f1': 47.1}","{'acc': 75.0, 'f1': 59.3}","{'acc': 64.6, 'f1': 58.3}","{'acc': 65.9, 'f1': 58.5}","{'acc': 64.5, 'f1': 57.6}","{'acc': 76.3, 'f1': 66.1}"
HateEval-eng,"{'acc': 49.5, 'f1': 49.5}","{'acc': 58.2, 'f1': 51.5}","{'acc': 64.0, 'f1': 62.1}","{'acc': 61.9, 'f1': 56.8}","{'acc': 62.1, 'f1': 57.6}","{'acc': 67.2, 'f1': 66.2}"
HateEval-spa,"{'acc': 42.6, 'f1': 38.8}","{'acc': 53.7, 'f1': 53.7}","{'acc': 49.9, 'f1': 49.8}","{'acc': 46.3, 'f1': 42.9}","{'acc': 47.2, 'f1': 43.3}","{'acc': 54.4, 'f1': 53.9}"
Gahd,"{'acc': 54.9, 'f1': 54.7}","{'acc': 45.1, 'f1': 37.8}","{'acc': 61.4, 'f1': 61.3}","{'acc': 56.9, 'f1': 55.3}","{'acc': 57.3, 'f1': 55.7}","{'acc': 57.9, 'f1': 56.8}"


Qwen2.5-14B

In [ ]:
model_group = "Qwen2.5-14B"
result = {}
probs = run_eval()


In [11]:
with open("Qwen14_probs.pkl", "rb") as f:
    probs = pickle.load(f)

In [12]:
report = make_report(probs, df_val)

In [13]:
report

,Human,Human-Lgb,Base,Lgb,Mean,Vote
HateXplain,"{'acc': 73.9, 'f1': 70.0}","{'acc': 75.6, 'f1': 73.1}","{'acc': 66.3, 'f1': 54.6}","{'acc': 73.2, 'f1': 69.5}","{'acc': 74.9, 'f1': 72.2}","{'acc': 73.8, 'f1': 72.4}"
GermEval18,"{'acc': 82.8, 'f1': 80.9}","{'acc': 83.2, 'f1': 80.5}","{'acc': 75.8, 'f1': 75.3}","{'acc': 82.6, 'f1': 79.9}","{'acc': 82.6, 'f1': 79.5}","{'acc': 80.1, 'f1': 75.0}"
Haternet,"{'acc': 69.5, 'f1': 67.3}","{'acc': 77.0, 'f1': 73.5}","{'acc': 46.4, 'f1': 46.3}","{'acc': 75.0, 'f1': 72.2}","{'acc': 77.4, 'f1': 74.0}","{'acc': 81.2, 'f1': 76.0}"
AHSD,"{'acc': 81.9, 'f1': 74.9}","{'acc': 79.5, 'f1': 72.9}","{'acc': 91.9, 'f1': 84.9}","{'acc': 72.8, 'f1': 67.1}","{'acc': 81.5, 'f1': 74.8}","{'acc': 63.8, 'f1': 59.6}"
AbusEval,"{'acc': 82.1, 'f1': 68.7}","{'acc': 82.1, 'f1': 66.2}","{'acc': 66.9, 'f1': 63.4}","{'acc': 80.5, 'f1': 67.1}","{'acc': 81.5, 'f1': 68.2}","{'acc': 81.4, 'f1': 67.2}"
GermEval19,"{'acc': 81.3, 'f1': 78.9}","{'acc': 82.2, 'f1': 79.1}","{'acc': 72.2, 'f1': 71.8}","{'acc': 81.2, 'f1': 78.3}","{'acc': 81.9, 'f1': 79.0}","{'acc': 79.5, 'f1': 74.0}"
ViHSD,"{'acc': 85.0, 'f1': 76.0}","{'acc': 86.9, 'f1': 77.5}","{'acc': 78.7, 'f1': 71.9}","{'acc': 87.3, 'f1': 75.9}","{'acc': 87.1, 'f1': 76.0}","{'acc': 86.4, 'f1': 70.9}"
HateEval-eng,"{'acc': 67.2, 'f1': 66.9}","{'acc': 70.0, 'f1': 69.4}","{'acc': 63.0, 'f1': 61.7}","{'acc': 70.0, 'f1': 69.8}","{'acc': 69.7, 'f1': 69.5}","{'acc': 69.0, 'f1': 68.3}"
HateEval-spa,"{'acc': 65.2, 'f1': 64.6}","{'acc': 67.5, 'f1': 67.4}","{'acc': 57.9, 'f1': 54.6}","{'acc': 67.2, 'f1': 67.0}","{'acc': 65.2, 'f1': 65.1}","{'acc': 67.8, 'f1': 67.8}"
Gahd,"{'acc': 77.7, 'f1': 77.7}","{'acc': 77.3, 'f1': 77.0}","{'acc': 73.5, 'f1': 73.5}","{'acc': 75.5, 'f1': 75.0}","{'acc': 75.8, 'f1': 75.0}","{'acc': 71.6, 'f1': 69.3}"
